In [5]:
import pandas as pd
import numpy as py


# Load CSV file from Google Drive
file_path = 'C:/Users/DELL/Documents/project_data/data/wfp_food_prices_eth.csv'
df = pd.read_csv(file_path)

print(df.columns)
print(df.head())

Index(['date', 'admin1', 'admin2', 'market', 'latitude', 'longitude',
       'category', 'commodity', 'unit', 'priceflag', 'pricetype', 'currency',
       'price', 'usdprice'],
      dtype='object')
         date       admin1      admin2            market  latitude  longitude  \
0       #date   #adm1+name  #adm2+name  #loc+market+name  #geo+lat   #geo+lon   
1  2000-01-15  Addis Ababa    AA ZONE1       Addis Ababa  9.024325  38.749226   
2  2000-01-15  Addis Ababa    AA ZONE1       Addis Ababa  9.024325  38.749226   
3  2000-01-15  Addis Ababa    AA ZONE1       Addis Ababa  9.024325  38.749226   
4  2000-01-15  Addis Ababa    AA ZONE1       Addis Ababa  9.024325  38.749226   

             category        commodity        unit         priceflag  \
0          #item+type       #item+name  #item+unit  #item+price+flag   
1  cereals and tubers    Maize (white)      100 KG            actual   
2  cereals and tubers  Sorghum (white)      100 KG            actual   
3  cereals and tubers     

In [6]:
# Drop first row, reset index, convert date column to datetime, and cast numeric columns
df = (
    df.drop(0)
      .reset_index(drop=True)
      .assign(date=lambda x: pd.to_datetime(x['date']))
      .astype({'price': float, 'usdprice': float})
)

# Sort by date, drop duplicates, reset index
df = df.sort_values('date').drop_duplicates().reset_index(drop=True)

# Extract year and month
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Clean commodity column: remove parentheses text and extract variety
df['commodity_base'] = df['commodity'].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()
df['variety'] = df['commodity'].str.extract(r'\((.*?)\)', expand=False).fillna('')

# Check missing values
print(df.isnull().sum())



date              0
admin1            5
admin2            5
market            0
latitude          5
longitude         5
category          0
commodity         0
unit              0
priceflag         0
pricetype         0
currency          0
price             0
usdprice          0
year              0
month             0
commodity_base    0
variety           0
dtype: int64


In [ ]:
import numpy as np
import pandas as pd

# Replace unavailable admin2 values with NaN
df['admin2'] = df['admin2'].replace('Administrative unit not available', np.nan)

# Convert latitude and longitude to numeric
df[['latitude', 'longitude']] = df[['latitude', 'longitude']].apply(pd.to_numeric, errors='coerce')

# Keep only rows with positive price
df = df[df['price'] > 0].reset_index(drop=True)

# Convert categorical columns
df[['admin1', 'admin2', 'market', 'pricetype', 'category']] = df[['admin1', 'admin2', 'market', 'pricetype', 'category']].astype('category')

# Handle missing admin2 values
df['admin2'] = df['admin2'].cat.add_categories('Unknown').fillna('Unknown')

# Drop rows with missing admin1, latitude, longitude
df = df.dropna(subset=['admin1', 'latitude', 'longitude']).reset_index(drop=True)

# Replace empty variety with 'Standard'
df['variety'] = df['variety'].replace('', 'Standard').fillna('Standard')

# Clean unit column
df['unit'] = df['unit'].str.strip().str.upper()

# Drop unnecessary columns
df = df.drop(columns=['commodity','priceflag','currency'])

# Final checks
print(df.shape)
print(df.tail())
print(df.head())
print(df.info())